In [20]:
import PySimpleGUI as sg
import pandas as pd
import time
import math
import random
import matplotlib.pyplot as plt  # <-- needed for plotting
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg


df = pd.read_csv("Planet Habitability Data - Sheet1.csv")

def hz_score(semi_major_axis, hz_range):
    try:
        hz_min, hz_max = map(float, str(hz_range).replace(' AU','').split('-'))
    except:
        return 0
    if hz_min <= semi_major_axis <= hz_max:
        return 100
    else:
        dist = min(abs(semi_major_axis - hz_min), abs(semi_major_axis - hz_max))
        return max(0, 100 - dist*50)

gas_scores = {'N2':100,'O2':100,'CO2':50,'H2O':90, 'H':20, 'He': 20, 'CH4':20}
def atmosphere_score(atmosphere_str):
    if pd.isna(atmosphere_str):
        return 0
    gases = [g.strip() for g in str(atmosphere_str).split(',')]
    scores = [gas_scores.get(g,0) for g in gases]
    return sum(scores)/len(scores) if scores else 0

def size_score(radius):
    if pd.isna(radius):
        return 0
    ideal_radius = 1
    diff = abs(radius - ideal_radius)
    return max(0, 100 - diff * 50)

def star_score(star_type):
    if pd.isna(star_type):
        return 0
    if star_type.startswith('G') or star_type.startswith('K'):
        return 100
    elif star_type.startswith('M'):
        return 50
    else:
        return 40

def calculate_habitability(planet_dict):
    weights = {'Habitable Zone':0.45,'Atmosphere':0.30,'Planet Size':0.15,'Star Type':0.10}
    return sum(planet_dict[key]*weights[key] for key in weights)

def draw_figure(canvas, figure):
    figure_canvas_agg = FigureCanvasTkAgg(figure, canvas)
    figure_canvas_agg.draw()
    figure_canvas_agg.get_tk_widget().pack(side='top', fill='both', expand=1)
    return figure_canvas_agg

def plot_habitability_window(df):
    planets = []
    scores = []
    sma = []
    in_hz_flag = []

    for idx, row in df.iterrows():
        hab_score = calculate_habitability({
            'Habitable Zone': hz_score(row['Orbital Semi-Major Axis [AU]'], row['Habitable Zone of Star']),
            'Atmosphere': atmosphere_score(row['Atmosphere']),
            'Planet Size': size_score(row['Planet Radius [Earth Radius]']),
            'Star Type': star_score(row['Star Type'])
        })

        planets.append(row['Planetary Name'])
        scores.append(hab_score)
        sma.append(row['Orbital Semi-Major Axis [AU]'])

        try:
            hz_min, hz_max = map(float, str(row['Habitable Zone of Star']).replace(' AU','').split('-'))
            in_hz_flag.append(1 if hz_min <= row['Orbital Semi-Major Axis [AU]'] <= hz_max else 0)
        except:
            in_hz_flag.append(0)

    fig, ax = plt.subplots(figsize=(10,6))
    
    scatter = ax.scatter(
        sma,
        scores,
        c=in_hz_flag,
        cmap='coolwarm',
        s=200
    )

    for i, name in enumerate(planets):
        ax.annotate(name, (sma[i], scores[i]), textcoords="offset points", xytext=(5,5))

    ax.set_xlabel("Distance From Host Star (AU)")
    ax.set_ylabel("Estimated Habitability Score")
    ax.set_title("Planet Habitability vs Distance From Host Star")

    cbar = plt.colorbar(scatter)
    cbar.set_label("In Habitable Zone (1 = Yes, 0 = No)")

    plt.tight_layout()

    plot_layout = [
        [sg.Canvas(key='-CANVAS-')],
        [sg.Button('Close')]
    ]

    plot_window = sg.Window("Habitability Plot", plot_layout, finalize=True)
    draw_figure(plot_window['-CANVAS-'].TKCanvas, fig)
    
def plot_star_type_window(df):
    star_types = []
    scores = []

    for idx, row, in df.iterrows():
        hab_score = calculate_habitability({
            'Habitable Zone': hz_score(row['Orbital Semi-Major Axis [AU]'], row['Habitable Zone of Star']),
            'Atmosphere': atmosphere_score(row['Atmosphere']),
            'Planet Size': size_score(row['Planet Radius [Earth Radius]']),
            'Star Type': star_score(row['Star Type'])
        })

        scores.append(hab_score)
        st = str(row['Star Type'])[0].upper()
        star_types.append(st)
    order = ["O", "B", "A", "F", "G", "K", "M"]
    x_vals = [order.index(t) if t in order else None for t in star_types]

    fig, ax = plt.subplots(figsize=(10,6))
    ax.scatter(x_vals, scores, s=200, c='red')

    for i, st in enumerate(star_types):
        ax.annotate(st, (x_vals[i], scores[i]), textcoords="offset points", xytext=(5,5))

    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_xlabel("Star Type")
    ax.set_ylabel("Habitability Score")
    ax.set_title("Star Type vs Habitability Score")
    ax.grid(True)
    plt.tight_layout()

    layout = [
        [sg.Canvas(key='-STAR_CANVAS-')],
        [sg.Button("Close")]
    ]
    win = sg.Window("Star Type Plot", layout, finalize=True)
    draw_figure(win['-STAR_CANVAS-'].TKCanvas, fig)


def animate_planet_nasa_crt(graph, score, planet_name):
    graph.erase()
    w, h = graph.get_size()
    cx, cy = w / 2, h / 2
    radius = min(w, h) / 4

    outline_color = 'green'
    fill_color = 'green'
    margin = 20

    steps = 80
    flicker_range = 30

    points = 100
    wireframe = []
    for i in range(points):
        angle1 = i * 2*math.pi / points
        angle2 = (i+1) * 2*math.pi / points
        x1 = cx + radius * math.cos(angle1)
        y1 = cy + radius * math.sin(angle1)
        x2 = cx + radius * math.cos(angle2)
        y2 = cy + radius * math.sin(angle2)
        wireframe.append(((x1, y1), (x2, y2)))

    target_height = 2 * radius * (score / 100)
    for step in range(steps + 1):
        graph.erase()
        graph.draw_text(planet_name, (cx, h - margin), color='white', font=('Courier', 20, 'bold'))
        for line in wireframe:
            graph.draw_line(line[0], line[1], color=outline_color)
        fill_height = target_height * step / steps
        for y_offset in range(int(fill_height)):
            y = cy - radius + y_offset
            dx = (radius**2 - (y - cy)**2)**0.5
            if random.randint(0, 100) > flicker_range:
                graph.draw_line((cx - dx, y), (cx + dx, y), color=fill_color)
        for glow in range(1,4):
            graph.draw_circle((cx, cy), radius + glow*3, line_color='dark green')
        graph.draw_text("Computing...", (cx, margin), color='green', font=('Courier', 20))
        window.refresh()
        time.sleep(0.05)

    graph.erase()
    
    for y_offset in range(int(target_height)):
        y = cy - radius + y_offset
        dx = (radius**2 - (y - cy)**2)**0.5
        graph.draw_line((cx - dx, y), (cx + dx, y), color=fill_color)
    
    for line in wireframe:
        graph.draw_line(line[0], line[1], color=outline_color)
    for glow in range(1,4):
        graph.draw_circle((cx, cy), radius + glow*3, line_color='dark green')

    graph.draw_text("Calculation Complete", (cx, margin), color='green', font=('Courier', 19, 'bold'))
    window.refresh()
    time.sleep(1.5)  

    graph.erase()
    for y_offset in range(int(target_height)):
        y = cy - radius + y_offset
        dx = (radius**2 - (y - cy)**2)**0.5
        graph.draw_line((cx - dx, y), (cx + dx, y), color=fill_color)
    for line in wireframe:
        graph.draw_line(line[0], line[1], color=outline_color)
    for glow in range(1,4):
        graph.draw_circle((cx, cy), radius + glow*3, line_color='dark green')

    graph.draw_text(f"{planet_name} Habitability: {score:.1f}%", (cx, cy), color='white', font=('Courier', 18, 'bold'))
    window.refresh()


sg.theme('DarkBlack')
layout = [
    [sg.Text('Choose Planet For Evaluation', text_color='green', font=('Courier',14))],
    [sg.Input(key='-IN-', expand_x=True, background_color='black', text_color='green'),
     sg.Listbox(sorted(df['Planetary Name'].tolist()), size=(20,10),
                key='-LIST-', enable_events=True,
                expand_x=True, expand_y=True,
                background_color='black', text_color='green')],
    [sg.Button('Submit')],
    [sg.Graph(canvas_size=(600,600), graph_bottom_left=(0,0), graph_top_right=(600,600),
              key='-GRAPH-', background_color='black', expand_x=True, expand_y=True)]
]

window = sg.Window('Habitability Calculator', layout, resizable=True, finalize=True)
window.maximize()


plot_button_added = False 
while True:
    event, values = window.read()
    if event == sg.WIN_CLOSED:
        break

    if event == '-LIST-':
        selected = values['-LIST-'][0]
        window['-IN-'].update(selected)

    if event == 'Submit':
        planet_name = values['-IN-']
        if planet_name not in df['Planetary Name'].values:
            sg.popup("Please select a planet from the options listed")
            continue

        planet_row = df[df['Planetary Name'] == planet_name].iloc[0]
        planet_dict = {
            'Habitable Zone': hz_score(planet_row['Orbital Semi-Major Axis [AU]'],
                                       planet_row['Habitable Zone of Star']),
            'Atmosphere': atmosphere_score(planet_row['Atmosphere']),
            'Planet Size': size_score(planet_row['Planet Radius [Earth Radius]']),
            'Star Type': star_score(planet_row['Star Type'])
        }
        score = calculate_habitability(planet_dict)
        animate_planet_nasa_crt(window['-GRAPH-'], score, planet_name)

        if not plot_button_added:
            window.extend_layout(window, [[
                sg.Button('Show Plot', key='-PLOT-'),
                sg.Button('Show Star Plot', key='-STAR_PLOT-')
    ]])
            plot_button_added = True


    
    if event == '-PLOT-':
        plot_habitability_window(df)

    if event == '-STAR_PLOT-':
        plot_star_type_window(df)


window.close()
